In [ ]:
import os
import kagglehub
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from torch.optim import Adam
from tqdm.auto import tqdm
from google.colab import drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
with open("/content/drive/MyDrive/CNN/save_trained/a7a2.txt", "w") as f:
    f.write("shit assss fucking bitch.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ------------------------------
# 1) DETECT GPU
# ------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


Using device: cuda
GPU: Tesla T4


In [ ]:

# ------------------------------
# 2) IMAGE PREPROCESSING
# ------------------------------
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor()
])

In [ ]:
# ------------------------------
# 3) DOWNLOAD DATASET
# ------------------------------
path = kagglehub.dataset_download(
    "shreyanshpatel1/130k-real-vs-fake-face"
)

print("Dataset path:", path)
images_path = path + "/images"


Using Colab cache for faster access to the '130k-real-vs-fake-face' dataset.
Dataset path: /kaggle/input/130k-real-vs-fake-face


In [ ]:
# ------------------------------
# 4) LOAD DATASET
# ------------------------------
dataset = datasets.ImageFolder(
    root=images_path,
    transform=transform
)

loader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

print("Classes:", dataset.classes)

Classes: ['fake', 'real']


In [ ]:
# ------------------------------
# 5) BUILD CNN MODEL
# ------------------------------
model = nn.Sequential(
    nn.Conv2d(3, 16, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),

    nn.Conv2d(16, 32, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),

    nn.Flatten(),
    nn.Linear(32 * 32 * 32, 2)
)
model = model.to(device)

In [ ]:
# ------------------------------
# 6) LOSS + OPTIMIZER
# ------------------------------
loss_function = nn.CrossEntropyLoss()
optimizer = Adam(model.parameters(), lr=0.01)

In [ ]:
# ---------------------------------------------------
# 4) LOAD DATASET
# ---------------------------------------------------
# imagefolder automatically gives labels:
# fake = 0 , real = 1 (or opposite)

dataset = datasets.ImageFolder(

    root=images_path,
    transform=transform
)

# Create batches
loader = DataLoader(

    dataset,
    batch_size=32,
    shuffle=True
)

print("Classes:", dataset.classes)

Classes: ['fake', 'real']


In [ ]:
# ---------------------------------------------------
# 5) BUILD CNN MODEL
# ---------------------------------------------------
# Very small CNN for simplicity.

model = nn.Sequential(

    # First convolution layer
    nn.Conv2d(3, 16, kernel_size=3, padding=1),

    # Activation function
    nn.ReLU(),

    # Reduce image size
    nn.MaxPool2d(2),

    # Second convolution layer
    nn.Conv2d(16, 32, kernel_size=3, padding=1),

    nn.ReLU(),

    nn.MaxPool2d(2),

    # Flatten image to vector
    nn.Flatten(),

    # Fully connected layer
    nn.Linear(32 * 32 * 32, 2)
)

In [ ]:
# ---------------------------------------------------
# 6) LOSS FUNCTION + OPTIMIZER
# ---------------------------------------------------

# Classification loss
loss_function = nn.CrossEntropyLoss()

# Optimizer updates model weights
optimizer = Adam(model.parameters(), lr=0.01)

In [ ]:
print(next(model.parameters()).device)
print(images.device)

cuda:0
cuda:0


In [ ]:
# ------------------------------
# 7) TRAIN MODEL
# ------------------------------
epochs = 3
model.train()

for epoch in range(epochs):
    total_loss = 0

    # tqdm shows % progress + ETA automatically
    progress_bar = tqdm(
        loader,
        desc=f"Epoch {epoch+1}/{epochs}",
        unit="batch"
    )

    for images, labels in progress_bar:
        # move batch to GPU
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = loss_function(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # show live batch loss
        progress_bar.set_postfix(loss=loss.item())

    print(f"Epoch {epoch+1} Total Loss: {total_loss:.4f}")


Epoch 1/3:   0%|          | 0/4175 [00:00<?, ?batch/s]

Epoch 1 Total Loss: 2878.6808


Epoch 2/3:   0%|          | 0/4175 [00:00<?, ?batch/s]

Epoch 2 Total Loss: 2878.6242


Epoch 3/3:   0%|          | 0/4175 [00:00<?, ?batch/s]

Epoch 3 Total Loss: 2878.6829


In [ ]:
# ------------------------------
# 8) SAVE MODEL
# ------------------------------
save_path = "/content/drive/MyDrive/CNN/save_trained"

torch.save(model.state_dict(), save_path)

print("Training finished.")
print("Model saved to:", save_path)

Training finished.
Model saved to: /content/drive/MyDrive/CNN/save_trained/cnn_model.pth
